In [3]:
# imports

import requests
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3

In [4]:
# Initialization

load_dotenv(override=True)

requests.get("http://localhost:11434").content
OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

DB = "prices.db"

In [5]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [6]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [7]:
get_ticket_price("Paris")

DATABASE TOOL CALLED: Getting price for Paris


'Ticket price to Paris is $899.0'

In [8]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}
tools = [{"type": "function", "function": price_function}]
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

In [10]:

def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model="qwen3:14b", messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7897
* To create a public link, set `share=True` in `launch()`.


In [12]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model="qwen3:14b", messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = ollama.chat.completions.create(model="qwen3:14b", messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [13]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [15]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7898
* To create a public link, set `share=True` in `launch()`.


In [17]:
# Some imports for handling images

import base64
from io import BytesIO
from PIL import Image

In [4]:
from diffusers import DiffusionPipeline
import torch

import torch

pipe = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float32
)

pipe.to("cpu")

def artist(city):
    prompt = (
        f"An image representing a vacation in {city}, "
        f"showing tourist spots and everything unique about {city}, "
        f"in a vibrant pop-art style"
    )

    image = pipe(prompt).images[0]
    return image

Loading pipeline components...: 100%|██████████| 7/7 [00:00<00:00,  7.57it/s]


In [ ]:
image = artist("New York City")
display(image)

 14%|█▍        | 7/50 [12:26<1:16:14, 106.38s/it]

In [ ]:
from kokoro import KPipeline
import soundfile as sf

pipeline = KPipeline(lang_code="a")  # American English

def talker(message):
    generator = pipeline(message)

    for _, _, audio in generator:
        sf.write("speech.wav", audio, 24000)

    return "speech.wav"

In [ ]:
def chat(history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]

    messages = [{"role": "system", "content": system_message}] + history

    response = ollama.chat.completions.create(
        model="qwen3:14b",
        messages=messages,
        tools=tools
    )

    cities = []
    image = None

    while response.choices[0].finish_reason == "tool_calls":

        message = response.choices[0].message

        responses, cities = handle_tool_calls_and_return_cities(message)

        messages.append(message)
        messages.extend(responses)

        response = ollama.chat.completions.create(
            model="qwen.3:14b",
            messages=messages,
            tools=tools
        )

    reply = response.choices[0].message.content

    history.append({
        "role": "assistant",
        "content": reply
    })

    # Open-source TTS (Kokoro)
    voice_path = talker(reply)

    # Open-source Image Generation (SDXL)
    if cities:
        image = artist(cities[0])

    return history, voice_path, image

In [ ]:
# Callbacks (along with the chat() function above)

def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

# UI definition

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

# Hooking up events to callbacks

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, audio_output, image_output]
    )

ui.launch(inbrowser=True, auth=("jaya", "shree"))